# 06 — Testing and CI

This notebook treats the validator and pipeline as software that must itself be tested.

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ROOT

WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab')

## Inspect the test architecture

The final suite contains unit, schema-contract, integration, and regression tests.

In [2]:
sorted(path.name for path in (ROOT / "tests").glob("test_*.py"))

['test_business_rules_unit.py',
 'test_order_schema.py',
 'test_phase3_validation.py',
 'test_phase4_business_rules.py',
 'test_phase5_typed_pipeline.py',
 'test_phase6_reliability.py']

## Run a focused test file

Use subprocess so the notebook demonstrates the exact command a developer would run.

In [3]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_business_rules_unit.py"], cwd=ROOT, check=False)

CompletedProcess(args=['C:\\Users\\Victus 16\\PycharmProjects\\pandera-data-quality-lab\\.venv\\Scripts\\python.exe', '-m', 'pytest', '-q', 'tests/test_business_rules_unit.py'], returncode=0)

## Coverage is a guardrail

The repository quality gate requires 90%, but coverage is not evidence that business assertions are correct.

In [4]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "--cov=pandera_lab", "--cov-report=term-missing"], cwd=ROOT, check=False)

CompletedProcess(args=['C:\\Users\\Victus 16\\PycharmProjects\\pandera-data-quality-lab\\.venv\\Scripts\\python.exe', '-m', 'pytest', '-q', '--cov=pandera_lab', '--cov-report=term-missing'], returncode=0)

## Inspect CI

Read `.github/workflows/ci.yml`. Identify which checks run on every Python version and which run once.

In [5]:
print((ROOT / ".github" / "workflows" / "ci.yml").read_text(encoding="utf-8"))

name: CI

on:
  push:
    branches: [main]
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

jobs:
  test-matrix:
    name: Python ${{ matrix.python-version }}
    runs-on: ubuntu-latest
    strategy:
      fail-fast: false
      matrix:
        python-version: ["3.10", "3.12", "3.14"]

    steps:
      - name: Checkout
        uses: actions/checkout@v7

      - name: Set up Python
        uses: actions/setup-python@v7
        with:
          python-version: ${{ matrix.python-version }}
          cache: pip
          cache-dependency-path: pyproject.toml

      - name: Install project
        run: |
          python -m pip install --upgrade pip
          python -m pip install -e ".[dev]"

      - name: Compile Python
        run: python -m compileall -q src tests examples scripts

      - name: Lint
        run: ruff check .

      - name: Run tests
        run: python -m pytest -q

  quality-gate:
    name: Coverage and package build
    runs-on: ubuntu-latest

    

## Reflection

1. Which failure should a unit test catch before Pandera integration?
2. Which failure requires a full pipeline test?
3. Why keep frozen historical schemas?
4. Why is a package build part of release confidence?